## Day 1: Grounding a Support Bot in Real FAQ Text

A software product's support inbox gets the same questions repeatedly.
A plain LLM will guess at an answer (hallucination risk, and it has never
seen this product's actual refund policy). RAG fixes this by retrieving
the relevant FAQ entry first, then generating an answer from it — the
"open-book exam" pattern.


In [ ]:
faq_documents = [
    {"id": "faq-refunds", "text": "Refunds are available within 30 days of purchase, minus a $5 processing fee."},
    {"id": "faq-rate-limit", "text": "The API allows 100 requests per minute on the free tier, 2000 on the paid tier."},
    {"id": "faq-password", "text": "Reset your password from Settings > Security > Reset Password."},
]

def ungrounded_answer(question: str) -> str:
    # Stands in for a plain LLM call with no retrieved context at all.
    return "I believe most SaaS products offer a 14-day refund window, but I am not certain."

def grounded_answer(question: str, retrieved: dict) -> str:
    # Stands in for an LLM call given ONLY the retrieved snippet as context.
    source_id = retrieved['id']
    source_text = retrieved['text']
    return f'According to {source_id}: "{source_text}"'

question = "How long do I have to request a refund?"
retrieved = faq_documents[0]  # Day 2/3 show how this lookup actually happens

print("Ungrounded:", ungrounded_answer(question))
print("Grounded:  ", grounded_answer(question, retrieved))


## Day 2: Ranking FAQ Entries With a Toy Embedding + Cosine Similarity

To find `retrieved` above automatically, we need a similarity search.
Here's a from-scratch character-trigram "embedding" (no network calls
needed) plus cosine similarity, used to rank FAQ entries against a
question.


In [ ]:
import hashlib, math

def toy_embed(text: str, dims: int = 64) -> list[float]:
    vec = [0.0] * dims
    text = text.lower().replace(" ", "_")
    for i in range(len(text) - 2):
        trigram = text[i:i + 3]
        bucket = int(hashlib.md5(trigram.encode()).hexdigest(), 16) % dims
        vec[bucket] += 1.0
    return vec

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

query = "how do I get my money back"
query_vec = toy_embed(query)

ranked = sorted(
    (
        (cosine_similarity(query_vec, toy_embed(doc["text"])), doc["id"])
        for doc in faq_documents
    ),
    reverse=True,
)
ranked


**Limitation to notice:** the query above ("get my money back") shares
almost no character trigrams with `faq-refunds` ("Refunds are
available..."), so the toy embedding may rank it no higher than an
unrelated entry — it's matching spelling, not meaning. A real, trained
embedding model would place "get my money back" and "refund" close
together despite the different wording.


## Day 3: Moving FAQ Search Into a Vector Database

A brute-force Python loop over every FAQ entry is fine for 3 entries, not
for 30,000 support-doc chunks. A vector database indexes embeddings for
fast top-k search. Below: the typical create/add/query shape, with a
pure-Python fallback for when no vector database library is installed.


In [ ]:
try:
    import vector_db_client as vdb  # hypothetical client; not a real import here
    HAS_VECTOR_DB = True
except ImportError:
    HAS_VECTOR_DB = False

def brute_force_search(query: str, top_k: int = 2):
    query_vec = toy_embed(query)
    scored = sorted(
        ((cosine_similarity(query_vec, toy_embed(d["text"])), d) for d in faq_documents),
        key=lambda pair: pair[0],
        reverse=True,
    )
    return scored[:top_k]

def search(query: str, top_k: int = 2):
    if HAS_VECTOR_DB:
        collection = vdb.get_collection("faq")
        return collection.query(query_embeddings=[toy_embed(query)], n_results=top_k)
    return brute_force_search(query, top_k)  # pure-Python fallback used in this notebook

search("how do I get my money back")


**Distance vs. similarity:** most vector databases return a *distance*
(lower = more relevant), the opposite direction from the cosine
*similarity* scores above (higher = more relevant) — always check which
convention a given query result uses before sorting.

At production scale, teams typically use a managed vector database
service, a self-hosted vector search engine, or a vector extension on a
database they already run — the create/add/query shape above stays
roughly the same across all of them.


## Day 4: Chunking Long Docs and Answering Only From Retrieved Context

A full API reference doc is too long (and too irrelevant, mostly) to send
on every question. We chunk it with overlap, retrieve only the relevant
chunks for a given question, and answer strictly from them — refusing to
answer when nothing relevant comes back.


In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

api_reference_doc = (
    "The API allows 100 requests per minute on the free tier. "
    "Paid tier accounts get 2000 requests per minute. "
    "Requests beyond the limit receive a 429 status code and should be retried with backoff."
)

doc_chunks = chunk_text(api_reference_doc)
len(doc_chunks), doc_chunks[0]


In [ ]:
def call_llm(prompt: str) -> str:
    # Placeholder for a real LLM call constrained to the given context.
    return "Free tier: 100 requests/min. Paid tier: 2000 requests/min."

def answer(question: str, top_k: int = 2, min_relevance: float = 0.05) -> dict:
    hits = brute_force_search(question, top_k=top_k)
    relevant = [(score, doc) for score, doc in hits if score >= min_relevance]

    if not relevant:
        return {"answer": "I don't know — nothing relevant was found in the FAQ.", "sources": []}

    context = "\n".join(f'[{doc["id"]}] {doc["text"]}' for _, doc in relevant)
    prompt = (
        "Answer using ONLY this context, and say 'I don't know' if it doesn't contain the answer.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    reply = call_llm(prompt)
    return {"answer": reply, "sources": [doc["id"] for _, doc in relevant]}

answer("what is the API rate limit on the paid tier?")
